# 05 | Retrieval validation and query study

This experiment studies PLSA retrieval across three query styles: direct keywords, paraphrases, and related domain terms. It uses six selected 20 Newsgroups categories and compares PLSA with TF-IDF/VSM and LSI.

The corpus is split into training (60%), validation (20%), and test (20%) sets. Training data fit the models. Validation MAP@10 selects PLSA's topic count and, for query-likelihood scoring, its background-smoothing weight. The held-out test set is used once for the selected settings. We also measure query vocabulary coverage and local runtime/memory use.


## 1. Settings and imports

The notebook loads the common processed corpus. The six-category subset, split, vocabulary, model grid, query bank, and output folder are defined here because they are decisions specific to this study.


In [ ]:
import json
import os
from pathlib import Path
import sys
import threading
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    import psutil
except ImportError:
    psutil = None
from IPython.display import display
from sklearn.model_selection import train_test_split

REPOSITORY_ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.evaluation import evaluate_ranked_retrieval
from src.lsi import fit_lsi
from src.plsa_model import train_plsa_restarts
from src.retrieval import cosine_scores, plsa_query_log_likelihood, background_word_probabilities
from src.text_processing import load_nlp, preprocess_texts
from src.tfidf import fit_tfidf

SEED = 42
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TEST_FRACTION = 0.20
SELECTED_CATEGORIES = [
    'comp.graphics', 'misc.forsale', 'rec.sport.baseball',
    'sci.med', 'sci.space', 'talk.politics.guns',
]
K_GRID = [3, 6, 10, 20]
BACKGROUND_GRID = [0.0, 0.01, 0.05]
N_RESTARTS = 3
MAX_ITER = 100
TOLERANCE = 1e-4
MAX_FEATURES = 3000
MIN_DF = 3
MAX_DF = 0.90
RETRIEVAL_K = 10
FOLD_IN_ITERATIONS = 100
DATA_PATH = REPOSITORY_ROOT / 'data' / '20news_preprocessed.npz'
OUTPUT_DIR = REPOSITORY_ROOT / 'results' / 'retrieval_validation_query_study'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Repository: {REPOSITORY_ROOT.resolve()}')
print(f'Results folder: {OUTPUT_DIR.resolve()}')
print(f'PLSA K grid: {K_GRID}; query-likelihood background weights: {BACKGROUND_GRID}')


## 2. Select the corpus and define the 18-query bank

Each selected category has three queries written in different styles. Relevance is category membership in the selected six-category corpus. Labels are used for the split and evaluation only, never as PLSA training targets.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run src/preprocessing.ipynb first.')
with np.load(DATA_PATH, allow_pickle=False) as corpus:
    all_documents = corpus['documents'].tolist()
    all_labels = corpus['labels'].astype(np.int64, copy=False)
    all_category_names = corpus['category_names'].tolist()

missing_categories = sorted(set(SELECTED_CATEGORIES) - set(all_category_names))
if missing_categories:
    raise ValueError(f'Selected categories are absent from the corpus: {missing_categories}')
selected_ids = [all_category_names.index(name) for name in SELECTED_CATEGORIES]
selected_mask = np.isin(all_labels, selected_ids)
documents = [doc for doc, keep in zip(all_documents, selected_mask) if keep]
labels = all_labels[selected_mask]
category_names = list(SELECTED_CATEGORIES)
category_queries = {
    'comp.graphics': [
        ('direct', 'computer graphics rendering image pixels'),
        ('paraphrase', 'raster visualization color drawing display'),
        ('related_terms', 'jpeg gif image resolution file format'),
    ],
    'misc.forsale': [
        ('direct', 'for sale price shipping offer'),
        ('paraphrase', 'selling item cost postage interested'),
        ('related_terms', 'available purchase asking price contact'),
    ],
    'rec.sport.baseball': [
        ('direct', 'baseball players team season game'),
        ('paraphrase', 'pitcher batter inning home run'),
        ('related_terms', 'major league baseball statistics runs'),
    ],
    'sci.med': [
        ('direct', 'medical doctor disease treatment health'),
        ('paraphrase', 'patient symptom diagnosis medication clinical'),
        ('related_terms', 'hospital therapy drug research study'),
    ],
    'sci.space': [
        ('direct', 'space NASA spacecraft orbit planet'),
        ('paraphrase', 'shuttle mission satellite launch moon'),
        ('related_terms', 'astronomy universe planetary exploration'),
    ],
    'talk.politics.guns': [
        ('direct', 'gun control firearms second amendment'),
        ('paraphrase', 'weapon legislation rights handgun rifle'),
        ('related_terms', 'firearm policy congress law'),
    ],
}
if set(category_queries) != set(category_names) or any(len(v) != 3 for v in category_queries.values()):
    raise ValueError('The query bank must contain three queries for each selected category.')
category_counts = pd.DataFrame({
    'category': category_names,
    'documents': [int(np.count_nonzero(labels == all_category_names.index(name))) for name in category_names],
})
print(f'Selected documents: {len(documents):,}; queries: {sum(map(len, category_queries.values()))}')
display(category_counts)


## 3. Create stratified train, validation, and test splits

The split proportions are 60/20/20. The first split holds out 40%; that remainder is divided evenly into validation and test. Count features are fitted on training documents only. Empty documents remain in the split records but are excluded from retrieval candidates for all methods.


In [ ]:
docs_train, docs_remaining, y_train, y_remaining = train_test_split(
    documents, labels, test_size=1 - TRAIN_FRACTION,
    random_state=SEED, stratify=labels,
)
relative_test_size = TEST_FRACTION / (VALIDATION_FRACTION + TEST_FRACTION)
docs_validation, docs_test, y_validation, y_test = train_test_split(
    docs_remaining, y_remaining, test_size=relative_test_size,
    random_state=SEED, stratify=y_remaining,
)

from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer(
    lowercase=False, tokenizer=str.split, token_pattern=None,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
    dtype=np.float64,
)
X_train = count_vectorizer.fit_transform(docs_train)
X_validation = count_vectorizer.transform(docs_validation)
X_test = count_vectorizer.transform(docs_test)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_validation = np.asarray([bool(text.strip()) for text in docs_validation])
nonempty_test = np.asarray([bool(text.strip()) for text in docs_test])

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'documents': [len(docs_train), len(docs_validation), len(docs_test)],
    'empty_documents': [
        sum(not text.strip() for text in docs_train),
        int((~nonempty_validation).sum()), int((~nonempty_test).sum()),
    ],
})
display(split_summary)
print('Count matrix shapes:', X_train.shape, X_validation.shape, X_test.shape)


## 4. Fit the TF-IDF and LSI reference models

The reference representations use the same train/validation/test text splits as PLSA. TF-IDF vocabulary and inverse document frequencies, plus LSI's SVD, are fitted on training text only.


In [ ]:
tfidf_vectorizer, (X_train_tfidf, X_validation_tfidf, X_test_tfidf) = fit_tfidf(
    docs_train, docs_validation, docs_test,
    max_features=MAX_FEATURES, min_df=MIN_DF, max_df=MAX_DF,
)
lsi_model, (lsi_train, lsi_validation, lsi_test) = fit_lsi(
    X_train_tfidf, X_validation_tfidf, X_test_tfidf,
    n_components=50, seed=SEED,
)
print('TF-IDF shapes:', X_train_tfidf.shape, X_validation_tfidf.shape, X_test_tfidf.shape)
print('LSI shapes:', lsi_train.shape, lsi_validation.shape, lsi_test.shape)


## 5. Prepare query representations and resource profiling

Query text uses the same lemmatization and stop-word removal as the corpus. Query counts, TF-IDF vectors, and LSI vectors are transformed using the already-fitted training representations. The profiling helper records wall time and, when `psutil` is available, process RSS before, after, and at the observed peak. Runtime and memory measurements depend on the machine and environment.


In [ ]:
query_nlp = load_nlp()
query_records = []
for category, variants in category_queries.items():
    for query_type, query_text in variants:
        processed = preprocess_texts([query_text], nlp=query_nlp)[0]
        query_records.append({
            'query_category': category,
            'target_label': all_category_names.index(category),
            'query_type': query_type, 'query_text': query_text,
            'processed_query': processed,
            'counts': count_vectorizer.transform([processed]).astype(np.float64),
            'tfidf': tfidf_vectorizer.transform([processed]),
        })
for record in query_records:
    record['lsi'] = lsi_model.transform(record['tfidf'])

process = psutil.Process(os.getpid()) if psutil is not None else None
def profile_resources(function, *args, **kwargs):
    if process is None:
        start = time.perf_counter()
        result = function(*args, **kwargs)
        elapsed = time.perf_counter() - start
        return result, {
            'runtime_seconds': round(elapsed, 3),
            'memory_before_MB': np.nan, 'memory_after_MB': np.nan,
            'peak_process_memory_MB': np.nan, 'peak_increase_from_start_MB': np.nan,
        }
    memory_samples = [process.memory_info().rss / (1024 ** 2)]
    stop_sampling = threading.Event()
    def sample_memory():
        while not stop_sampling.is_set():
            try:
                memory_samples.append(process.memory_info().rss / (1024 ** 2))
            except psutil.Error:
                pass
            stop_sampling.wait(0.05)
    memory_before = process.memory_info().rss / (1024 ** 2)
    sampler = threading.Thread(target=sample_memory, daemon=True)
    start = time.perf_counter()
    sampler.start()
    try:
        result = function(*args, **kwargs)
    finally:
        elapsed = time.perf_counter() - start
        stop_sampling.set()
        sampler.join()
        memory_after = process.memory_info().rss / (1024 ** 2)
    return result, {
        'runtime_seconds': round(elapsed, 3),
        'memory_before_MB': round(memory_before, 2),
        'memory_after_MB': round(memory_after, 2),
        'peak_process_memory_MB': round(max(memory_samples), 2),
        'peak_increase_from_start_MB': round(max(memory_samples) - memory_before, 2),
    }
print('Prepared query representations:', len(query_records))


## 6. Fit PLSA models across the validation grid

We fit PLSA at each predeclared K using three random initializations and retain the restart with the highest training log-likelihood. Validation and test labels do not participate in fitting. The training-resource measurements wrap these same fits, so no extra profiling-only model run is needed.


In [ ]:
models_by_k = {}
training_resource_rows = []
restart_rows = []
for k_topics in K_GRID:
    print(f'\nFitting K={k_topics} with {N_RESTARTS} restarts')
    (model, runs), measurements = profile_resources(
        train_plsa_restarts, X_train, n_topics=k_topics,
        n_restarts=N_RESTARTS, max_iter=MAX_ITER, tol=TOLERANCE,
        base_seed=SEED + 100 * k_topics,
    )
    models_by_k[k_topics] = model
    training_resource_rows.append({
        'K': k_topics, 'selected_iterations': model.n_iter_,
        'best_training_log_likelihood': float(model.log_likelihoods[-1]),
        **measurements,
    })
    for restart, (restart_model, log_likelihood) in enumerate(runs, start=1):
        restart_rows.append({
            'K': k_topics, 'restart': restart,
            'seed': SEED + 100 * k_topics + restart - 1,
            'iterations': restart_model.n_iter_,
            'training_log_likelihood': float(log_likelihood),
            'selected': restart_model is model,
        })
training_resources = pd.DataFrame(training_resource_rows)
restart_summary = pd.DataFrame(restart_rows)
print('Training runtime and memory by K:')
display(training_resources.round(3))


## 7. Tune PLSA settings on validation

For each K, we infer validation document-topic mixtures once. PLSA cosine does not use a background weight, so it is compared across K only. Query likelihood is evaluated at each background weight in the grid. The background distribution is estimated from training counts only.

The primary selection criterion is macro MAP@10 across all 18 validation queries. Precision@10 breaks ties; remaining ties prefer the smaller K, and for query likelihood then the smaller background weight. TF-IDF and LSI are evaluated on validation as fixed references, not tuned.


In [ ]:
background_probability = background_word_probabilities(X_train)
metric_columns = ['Precision@10', 'Recall@10', 'AP@10', 'NDCG@10']

def metric_row(scores, labels_for_split, record, method, split_name,
               k_topics=None, background_weight=np.nan):
    values = evaluate_ranked_retrieval(
        scores, labels_for_split, record['target_label'], k=RETRIEVAL_K,
    )
    return {
        'split': split_name, 'K': k_topics, 'background_weight': background_weight,
        'query_category': record['query_category'], 'query_type': record['query_type'],
        'query_text': record['query_text'], 'method': method,
        'Precision@10': values['Precision@k'], 'Recall@10': values['Recall@k'],
        'AP@10': values['AP@k'], 'NDCG@10': values['NDCG@k'],
    }

validation_plsa_rows = []
for k_topics, model in models_by_k.items():
    theta_validation = model.fold_in(X_validation, max_iter=FOLD_IN_ITERATIONS)
    eligible_theta = theta_validation[nonempty_validation]
    eligible_labels = y_validation[nonempty_validation]
    for record in query_records:
        q_theta = model.fold_in(record['counts'], max_iter=FOLD_IN_ITERATIONS)
        cosine = cosine_scores(q_theta, eligible_theta)
        validation_plsa_rows.append(metric_row(
            cosine, eligible_labels, record, 'PLSA cosine', 'validation', k_topics,
        ))
        for weight in BACKGROUND_GRID:
            likelihood = plsa_query_log_likelihood(
                record['counts'], eligible_theta, model.phi_,
                background_probability=background_probability,
                background_weight=weight, smoothing=1e-12,
            )
            validation_plsa_rows.append(metric_row(
                likelihood, eligible_labels, record, 'PLSA query likelihood',
                'validation', k_topics, weight,
            ))
validation_plsa = pd.DataFrame(validation_plsa_rows)

validation_baseline_rows = []
for record in query_records:
    for method, scores in {
        'TF-IDF/VSM': cosine_scores(record['tfidf'], X_validation_tfidf[nonempty_validation]),
        'LSI': cosine_scores(record['lsi'], lsi_validation[nonempty_validation]),
    }.items():
        validation_baseline_rows.append(metric_row(
            scores, y_validation[nonempty_validation], record, method, 'validation',
        ))
validation_baselines = pd.DataFrame(validation_baseline_rows)

cosine_tuning = (
    validation_plsa[validation_plsa['method'] == 'PLSA cosine']
    .groupby('K', as_index=False)[metric_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
likelihood_tuning = (
    validation_plsa[validation_plsa['method'] == 'PLSA query likelihood']
    .groupby(['K', 'background_weight'], as_index=False)[metric_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
best_cosine = cosine_tuning.sort_values(
    ['MAP@10', 'Precision@10', 'K'], ascending=[False, False, True],
).iloc[0].to_dict()
best_likelihood = likelihood_tuning.sort_values(
    ['MAP@10', 'Precision@10', 'K', 'background_weight'],
    ascending=[False, False, True, True],
).iloc[0].to_dict()
validation_baseline_summary = (
    validation_baselines.groupby('method', as_index=False)[metric_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
print('Validation tuning for PLSA cosine:')
display(cosine_tuning.sort_values('K').round(4))
print('Validation tuning for PLSA query likelihood:')
display(likelihood_tuning.sort_values(['K', 'background_weight']).round(4))
print('Selected PLSA settings:')
display(pd.DataFrame([
    {'method': 'PLSA cosine', 'selected_K': int(best_cosine['K']), 'selected_background_weight': np.nan,
     'validation_MAP@10': best_cosine['MAP@10']},
    {'method': 'PLSA query likelihood', 'selected_K': int(best_likelihood['K']),
     'selected_background_weight': best_likelihood['background_weight'],
     'validation_MAP@10': best_likelihood['MAP@10']},
]))
print('Validation reference results:')
display(validation_baseline_summary.round(4))


## 8. Evaluate the selected settings on the held-out test set

After validation selects the PLSA settings, we apply each selected model and scoring rule to the test set. The same 18 queries, preprocessing, relevance labels, and non-empty test candidates are used for PLSA, TF-IDF, and LSI. Test metrics are not used to revise the selected settings.


In [ ]:
cosine_model = models_by_k[int(best_cosine['K'])]
likelihood_model = models_by_k[int(best_likelihood['K'])]
(theta_test_cosine, cosine_inference_measurements) = profile_resources(
    cosine_model.fold_in, X_test, max_iter=FOLD_IN_ITERATIONS,
)
(theta_test_likelihood, likelihood_inference_measurements) = profile_resources(
    likelihood_model.fold_in, X_test, max_iter=FOLD_IN_ITERATIONS,
)
inference_resources = pd.DataFrame([
    {'operation': 'Fold in test documents for validation-selected PLSA cosine model',
     'K': cosine_model.n_topics, **cosine_inference_measurements},
    {'operation': 'Fold in test documents for validation-selected PLSA query-likelihood model',
     'K': likelihood_model.n_topics, **likelihood_inference_measurements},
])

y_test_eval = y_test[nonempty_test]
test_rows = []
for record in query_records:
    for method, scores in {
        'TF-IDF/VSM': cosine_scores(record['tfidf'], X_test_tfidf[nonempty_test]),
        'LSI': cosine_scores(record['lsi'], lsi_test[nonempty_test]),
        'PLSA cosine': cosine_scores(
            cosine_model.fold_in(record['counts'], max_iter=FOLD_IN_ITERATIONS),
            theta_test_cosine[nonempty_test],
        ),
        'PLSA query likelihood': plsa_query_log_likelihood(
            record['counts'], theta_test_likelihood[nonempty_test],
            likelihood_model.phi_, background_probability=background_probability,
            background_weight=float(best_likelihood['background_weight']),
            smoothing=1e-12,
        ),
    }.items():
        test_rows.append(metric_row(
            scores, y_test_eval, record, method, 'test',
            int(best_cosine['K']) if method == 'PLSA cosine' else (
                int(best_likelihood['K']) if method == 'PLSA query likelihood' else np.nan
            ),
            float(best_likelihood['background_weight']) if method == 'PLSA query likelihood' else np.nan,
        ))
test_metrics = pd.DataFrame(test_rows)
macro_columns = ['Precision@10', 'Recall@10', 'AP@10', 'NDCG@10']
test_summary = (
    test_metrics.groupby('method', sort=False)[macro_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
query_family_summary = (
    test_metrics.groupby(['query_type', 'method'], as_index=False)[macro_columns].mean()
    .rename(columns={'AP@10': 'MAP@10'})
)
print('Held-out test macro-average:')
display(test_summary.round(4))
print('Held-out test results by query type:')
display(query_family_summary.round(4))
print('Inference runtime and memory:')
display(inference_resources.round(3))


## 9. Check query vocabulary coverage

A query term outside the training vocabulary cannot contribute to count-based PLSA likelihood or the fitted TF-IDF/LSI representations. Coverage helps explain query-level variation, especially for paraphrases and related-term queries. This is a descriptive diagnostic; it does not change the rankings or tuning decisions.


In [ ]:
training_vocabulary = set(count_vectorizer.vocabulary_)
coverage_rows = []
for record in query_records:
    tokens = record['processed_query'].split()
    unique_terms = set(tokens)
    matched = unique_terms & training_vocabulary
    coverage_rows.append({
        'query_category': record['query_category'],
        'query_type': record['query_type'], 'query_text': record['query_text'],
        'processed_query': record['processed_query'],
        'unique_query_terms': len(unique_terms),
        'terms_in_training_vocabulary': len(matched),
        'terms_oov': len(unique_terms - training_vocabulary),
        'vocabulary_coverage': len(matched) / len(unique_terms) if unique_terms else 0.0,
    })
query_coverage = pd.DataFrame(coverage_rows)
per_query_test = test_metrics.merge(
    query_coverage[['query_category', 'query_type', 'query_text',
                    'unique_query_terms', 'terms_in_training_vocabulary',
                    'terms_oov', 'vocabulary_coverage']],
    on=['query_category', 'query_type', 'query_text'],
    how='left', validate='many_to_one',
)
print('Query vocabulary coverage:')
display(query_coverage.sort_values(['query_category', 'query_type']).round(4))
print('Test MAP@10 for related-term queries:')
display(per_query_test[per_query_test['query_type'] == 'related_terms'].pivot(
    index='query_category', columns='method', values='AP@10',
).round(4))


## 10. Visualize the findings

The charts compare the validation tuning surface and held-out query-family performance. Validation plots show the settings considered; test plots summarize the selected PLSA configurations alongside the fixed TF-IDF and LSI references.


In [ ]:
validation_fig, validation_axes = plt.subplots(1, 2, figsize=(12, 4.5))
validation_axes[0].plot(cosine_tuning['K'], cosine_tuning['MAP@10'], marker='o')
validation_axes[0].axvline(int(best_cosine['K']), linestyle='--', alpha=0.5, label=f"Selected K={int(best_cosine['K'])}")
validation_axes[0].set(title='PLSA cosine: validation MAP@10', xlabel='K', ylabel='MAP@10')
validation_axes[0].set_xticks(K_GRID)
validation_axes[0].grid(alpha=0.3)
validation_axes[0].legend()
for weight, group in likelihood_tuning.groupby('background_weight'):
    validation_axes[1].plot(group['K'], group['MAP@10'], marker='o', label=f'background weight={weight:g}')
validation_axes[1].axvline(int(best_likelihood['K']), linestyle='--', alpha=0.5)
validation_axes[1].set(title='PLSA query likelihood: validation MAP@10', xlabel='K', ylabel='MAP@10')
validation_axes[1].set_xticks(K_GRID)
validation_axes[1].grid(alpha=0.3)
validation_axes[1].legend()
validation_fig.tight_layout()

test_fig, test_axes = plt.subplots(1, 2, figsize=(12, 4.5))
test_axes[0].bar(
    test_summary.index.astype(str), test_summary['MAP@10'].to_numpy(),
    color=list(plt.cm.tab10.colors[:len(test_summary)]),
)
test_axes[0].set(title='Held-out test MAP@10', ylabel='Macro-average AP@10', xlabel='Method', ylim=(0, 1))
test_axes[0].tick_params(axis='x', rotation=30)
test_axes[0].grid(axis='y', alpha=0.3)
family_plot = query_family_summary.pivot(index='query_type', columns='method', values='MAP@10')
family_plot.plot(kind='bar', ax=test_axes[1], rot=0)
test_axes[1].set(title='Held-out MAP@10 by query type', ylabel='Macro-average AP@10', xlabel='Query type', ylim=(0, 1))
test_axes[1].grid(axis='y', alpha=0.3)
test_axes[1].legend(fontsize=8)
test_fig.tight_layout()


## 11. Save outputs

Tuning tables, test metrics, query coverage, resource measurements, figures, and the run configuration are saved in `results/retrieval_validation_query_study/`. Rerunning this notebook replaces the previous outputs for this experiment.


In [ ]:
training_resources.to_csv(OUTPUT_DIR / 'plsa_training_resources.csv', index=False)
inference_resources.to_csv(OUTPUT_DIR / 'plsa_inference_resources.csv', index=False)
restart_summary.to_csv(OUTPUT_DIR / 'plsa_restart_summary.csv', index=False)
cosine_tuning.to_csv(OUTPUT_DIR / 'validation_plsa_cosine_tuning.csv', index=False)
likelihood_tuning.to_csv(OUTPUT_DIR / 'validation_plsa_query_likelihood_tuning.csv', index=False)
validation_baseline_summary.to_csv(OUTPUT_DIR / 'validation_baseline_summary.csv', index=False)
test_metrics.to_csv(OUTPUT_DIR / 'test_metrics_by_query.csv', index=False)
test_summary.to_csv(OUTPUT_DIR / 'test_metrics_macro_average.csv')
query_family_summary.to_csv(OUTPUT_DIR / 'test_metrics_by_query_type.csv', index=False)
query_coverage.to_csv(OUTPUT_DIR / 'query_vocabulary_coverage.csv', index=False)
per_query_test.to_csv(OUTPUT_DIR / 'test_metrics_with_query_coverage.csv', index=False)
validation_fig.savefig(OUTPUT_DIR / 'validation_tuning_map.png', dpi=160, bbox_inches='tight')
test_fig.savefig(OUTPUT_DIR / 'test_retrieval_by_query_type.png', dpi=160, bbox_inches='tight')

configuration = {
    'experiment': 'Retrieval validation and query study',
    'selected_categories': SELECTED_CATEGORIES,
    'query_types': ['direct', 'paraphrase', 'related_terms'],
    'seed': SEED,
    'split_fractions': {'train': TRAIN_FRACTION, 'validation': VALIDATION_FRACTION, 'test': TEST_FRACTION},
    'k_grid': K_GRID, 'background_weight_grid': BACKGROUND_GRID,
    'restarts_per_k': N_RESTARTS, 'max_iter': MAX_ITER, 'tolerance': TOLERANCE,
    'max_features': MAX_FEATURES, 'min_df': MIN_DF, 'max_df': MAX_DF,
    'retrieval_k': RETRIEVAL_K,
    'selection_metric': 'validation macro MAP@10; tie-break by Precision@10, then lower K and lower background weight',
    'validation_selected_plsa_cosine': {'K': int(best_cosine['K'])},
    'validation_selected_plsa_query_likelihood': {
        'K': int(best_likelihood['K']),
        'background_weight': float(best_likelihood['background_weight']),
    },
    'query_bank': category_queries,
    'documents': len(documents),
}
with (OUTPUT_DIR / 'configuration.json').open('w', encoding='utf-8') as handle:
    json.dump(configuration, handle, indent=2)
print(f'Saved retrieval validation/query outputs to {OUTPUT_DIR.resolve()}')
